# Exercício 2 - CRUD - Controle de Estoque - Consumir Insumo (UPDATE) <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/microsoftsqlserver/microsoftsqlserver-plain.svg" height="45" />🍽️

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![pyodbc](https://img.shields.io/badge/pyodbc-0078D4?style=flat-square)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-exercício%20%7C%20crud%20%7C%20estoque-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc%20%7C%20funções-purple)
![Biblioteca](https://img.shields.io/badge/requer-pyodbc-orange)

> A função `consumir_insumo()`: dá baixa numa quantidade usada na produção — o `UPDATE` do dia a dia. A regra de negócio importante aqui: **nunca deixar a quantidade ficar negativa**, então a função recusa o consumo se não houver estoque suficiente.

## 📋 Conteúdo

1. [Conectando](#-1-conectando)
2. [Construindo consumir_insumo()](#-2-construindo-consumir_insumo)
3. [Consumo Normal](#-3-consumo-normal)
4. [Tentando Consumir Mais do que Existe](#-4-tentando-consumir-mais-do-que-existe)


## 🔌 1. Conectando

In [1]:
from conexao import nova_conexao_sqlserver
from cores import *

conexao = nova_conexao_sqlserver(banco="HashtagCursoSQL", autocommit=True)
cursor = conexao.cursor()


## 🛠️ 2. Construindo consumir_insumo()

| Passo 🔑 | O que faz 🔓 |
|---|---|
| 1 | Busca a quantidade atual do insumo |
| 2 | Confere se tem o suficiente pra atender o consumo pedido |
| 3 | Se tiver, faz `UPDATE` subtraindo; se não tiver, recusa e avisa |

In [2]:
def consumir_insumo(nome, quantidade_usada):
    cursor.execute("SELECT Id, Quantidade FROM dbo.Estoque WHERE Insumo = ?", nome)
    insumo = cursor.fetchone()

    if not insumo:
        print(f"{VermelhoClaro}'{nome}' não está cadastrado no estoque.{Reset}")
        return False

    quantidade_atual = float(insumo.Quantidade)
    if quantidade_usada > quantidade_atual:
        print(f"{VermelhoClaro}Estoque insuficiente de '{nome}':{Reset} {AmareloClaro}tem {quantidade_atual}, precisa de {quantidade_usada}{Reset}")
        return False

    nova_quantidade = quantidade_atual - quantidade_usada
    cursor.execute(
        "UPDATE dbo.Estoque SET Quantidade = ?, AtualizadoEm = GETDATE() WHERE Id = ?",
        nova_quantidade, insumo.Id
    )
    print(f"{VerdeClaro}Consumido {quantidade_usada} de '{nome}'{Reset} — restam {MagentaClaro}{nova_quantidade}{Reset}")
    return True


## 🍰 3. Consumo Normal

Simulando uma produção que usa alguns insumos do estoque.

In [3]:
consumir_insumo("Farinha de Trigo", 10.0)
consumir_insumo("Ovos", 24.0)
consumir_insumo("Chocolate em Pó", 3.0)


Consumido 10.0 de 'Farinha de Trigo' — restam 60.0
Consumido 24.0 de 'Ovos' — restam 276.0
Consumido 3.0 de 'Chocolate em Pó' — restam 9.0


True

## 🚫 4. Tentando Consumir Mais do que Existe

O `Morango` tem só 8 kg — pedir 50 kg deveria ser recusado, sem deixar a quantidade ir negativa.

In [4]:
cursor.execute("SELECT Quantidade FROM dbo.Estoque WHERE Insumo = ?", "Morango")
antes = cursor.fetchone().Quantidade
print(f"{CinzaClaro}Morango antes da tentativa:{Reset} {MagentaClaro}{antes}{Reset}")

consumir_insumo("Morango", 50.0)

cursor.execute("SELECT Quantidade FROM dbo.Estoque WHERE Insumo = ?", "Morango")
depois = cursor.fetchone().Quantidade
print(f"{CinzaClaro}Morango depois da tentativa:{Reset} {MagentaClaro}{depois}{Reset} {CinzaEscuro}(não mudou){Reset}")

conexao.close()


Morango antes da tentativa: 8.00
Estoque insuficiente de 'Morango': tem 8.0, precisa de 50.0
Morango depois da tentativa: 8.00 (não mudou)


`consumir_insumo()` fecha as três operações de escrita do sistema (criar/somar, remover, consumir), sempre respeitando a regra de nunca deixar o estoque ficar negativo. Falta só a leitura — consultar o que tem, e identificar o que está acabando.

> ▶️ Próximo notebook: **Exercício 2 - CRUD - Controle de Estoque - Procurar Insumo (READ)**.